# Assignment: Overfitting & Robustness — Building Strategies that Survive Reality
**From Backtest to Battle-Tested**

| Part | Focus | Methods |
|------|-------|---------|
| 1 | Walk-Forward Analysis | Rolling IS/OOS windows, regime capture |
| 2 | Regularisation | Lasso (L1), Ridge (L2), Elastic Net |
| 3 | Parameter Stability | Sensitivity surface, Flat-Top principle, Monte Carlo permutation |
| 4 | Statistical Testing | PSR, DSR, White's Reality Check, Haircut Sharpe |
| 5 | Monitoring & Kill Switch | Performance Z-score, drawdown triggers |

> *All data is fully simulated; no external feeds required.*

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import ndtr          # fast normal CDF
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import plotly.graph_objects as go
import plotly.subplots as ps
import warnings
warnings.filterwarnings('ignore')

SEED   = 42
rng    = np.random.default_rng(SEED)
ANNUAL = 252

# ── Simulated strategy universe ───────────────────────────────────────────────
T      = 1260   # 5 years daily
dates  = pd.date_range('2019-01-02', periods=T, freq='B')

# True daily alpha signal (mean-reverting, weakly persistent)
TRUE_ALPHA   = 0.0003          # 7.5 % annualised
DAILY_VOL    = 0.010           # 16 % ann
SIGNAL_DECAY = 0.92            # AR(1) coefficient of the latent signal

latent = np.zeros(T)
for t in range(1, T):
    latent[t] = SIGNAL_DECAY * latent[t-1] + rng.normal(0, 0.01)

# Returns = alpha * sign(latent) + vol * noise
true_signal = np.sign(latent)
returns     = TRUE_ALPHA * true_signal + DAILY_VOL * rng.standard_normal(T)
ret_series  = pd.Series(returns, index=dates, name='strategy')

print(f'Simulated {T} daily bars | ann. alpha ~ {TRUE_ALPHA*ANNUAL:.1%} | vol ~ {DAILY_VOL*ANNUAL**0.5:.1%}')

Simulated 1260 daily bars | ann. alpha ~ 7.6% | vol ~ 15.9%


---
## Part 1 | Walk-Forward Analysis

Walk-forward analysis mimics live deployment: we train on a rolling window, then validate on the *next* unseen block.  
This gives a **realistic OOS equity curve** assembled from non-overlapping test segments.

We simulate a simple **moving-average crossover strategy** whose lookback window is optimised in-sample each fold:

$$\text{Signal}_t = \text{sign}\!\left(\text{MA}_{\text{fast},t} - \text{MA}_{\text{slow},t}\right)$$

The in-sample Sharpe across candidate `(fast, slow)` pairs selects the best window; that window is then applied — **unchanged** — to the OOS segment.

In [2]:
# ── MA-crossover helper ───────────────────────────────────────────────────────
def ma_sharpe(rets, fast, slow):
    """Annualised Sharpe of a MA-crossover on a return series."""
    price  = (1 + rets).cumprod()
    ma_f   = price.rolling(fast).mean()
    ma_s   = price.rolling(slow).mean()
    signal = np.sign(ma_f - ma_s).shift(1)  # 1-bar lag avoids look-ahead
    pnl    = (signal * rets).dropna()
    if pnl.std() == 0:
        return -99
    return pnl.mean() / pnl.std() * ANNUAL**0.5

# ── Walk-forward grid ─────────────────────────────────────────────────────────
IS_BARS  = 252   # 1 year in-sample
OOS_BARS = 63    # 1 quarter OOS

FAST_GRID = [3, 5, 8, 10, 15]
SLOW_GRID = [20, 30, 40, 60, 80]

folds, is_sharpes, oos_sharpes, best_params = [], [], [], []
oos_pnl_series = pd.Series(dtype=float)

start = 0
fold  = 0
while start + IS_BARS + OOS_BARS <= T:
    is_ret  = ret_series.iloc[start : start + IS_BARS]
    oos_ret = ret_series.iloc[start + IS_BARS : start + IS_BARS + OOS_BARS]

    # Optimise in-sample
    best_sr, best_f, best_s = -99, FAST_GRID[0], SLOW_GRID[-1]
    for f in FAST_GRID:
        for s in SLOW_GRID:
            if f >= s:
                continue
            sr = ma_sharpe(is_ret, f, s)
            if sr > best_sr:
                best_sr, best_f, best_s = sr, f, s

    # Apply to OOS
    price_oos = (1 + pd.concat([is_ret.tail(best_s), oos_ret])).cumprod()
    ma_f_oos  = price_oos.rolling(best_f).mean()
    ma_s_oos  = price_oos.rolling(best_s).mean()
    sig_oos   = np.sign(ma_f_oos - ma_s_oos).shift(1).reindex(oos_ret.index)
    pnl_oos   = (sig_oos * oos_ret).dropna()
    oos_sr    = pnl_oos.mean() / pnl_oos.std() * ANNUAL**0.5 if pnl_oos.std() > 0 else 0

    folds.append(fold + 1)
    is_sharpes.append(best_sr)
    oos_sharpes.append(oos_sr)
    best_params.append((best_f, best_s))
    oos_pnl_series = pd.concat([oos_pnl_series, pnl_oos])

    start += OOS_BARS
    fold  += 1

print(f'{fold} folds | mean IS SR={np.mean(is_sharpes):.2f} | mean OOS SR={np.mean(oos_sharpes):.2f}')
print(f'IS→OOS degradation: {(np.mean(is_sharpes)-np.mean(oos_sharpes))/np.mean(is_sharpes):.1%}')

16 folds | mean IS SR=2.09 | mean OOS SR=1.45
IS→OOS degradation: 30.8%


In [3]:
# ── Plot: IS vs OOS Sharpe per fold + OOS equity curve ───────────────────────
fig = ps.make_subplots(
    rows=2, cols=1, vertical_spacing=0.12,
    subplot_titles=[
        'Walk-Forward: In-Sample vs Out-of-Sample Sharpe per Fold',
        'Walk-Forward OOS Equity Curve (assembled from non-overlapping segments)'
    ]
)

fig.add_trace(go.Bar(x=folds, y=is_sharpes,  name='IS Sharpe',
                     marker_color='#00d4aa', opacity=0.75), row=1, col=1)
fig.add_trace(go.Bar(x=folds, y=oos_sharpes, name='OOS Sharpe',
                     marker_color='#ff6b6b', opacity=0.85), row=1, col=1)
fig.add_hline(y=0, line_color='white', line_width=1, row=1, col=1)

oos_eq = (1 + oos_pnl_series).cumprod()
fig.add_trace(go.Scatter(x=oos_eq.index, y=oos_eq.values,
                          mode='lines', name='OOS Equity',
                          line=dict(color='#00d4aa', width=2)), row=2, col=1)
fig.add_hline(y=1, line_color='grey', line_dash='dot', row=2, col=1)

fig.update_layout(
    template='plotly_dark', height=600,
    title='Walk-Forward Analysis',
    barmode='group',
    legend=dict(x=0.01, y=0.97)
)
fig.update_xaxes(title_text='Fold', row=1, col=1)
fig.update_yaxes(title_text='Sharpe Ratio', row=1, col=1)
fig.update_yaxes(title_text='Portfolio Value', row=2, col=1)
fig.show()

**Key observation:** IS Sharpe is systematically higher than OOS Sharpe — the classic overfitting gap.  
The assembled OOS equity curve provides an unbiased estimate of live performance.

---
## Part 2 | Regularisation: Lasso, Ridge & Elastic Net

We build a **factor regression** problem: predict next-bar returns from 50 candidate features (30 are pure noise, 20 carry weak signal).  
We compare OLS vs regularised models on a held-out test set.

| Method | Penalty | Effect |
|--------|---------|--------|
| OLS | None | Overfits noise features |
| Ridge (L2) | $\lambda\|\beta\|_2^2$ | Shrinks all coefficients equally |
| Lasso (L1) | $\lambda\|\beta\|_1$ | Zeros out irrelevant features |
| Elastic Net | $\lambda_1\|\beta\|_1 + \lambda_2\|\beta\|_2^2$ | Hybrid: sparse + stable |

In [4]:
# ── Synthetic factor regression dataset ──────────────────────────────────────
N_SAMPLES  = 800
N_FEATURES = 50
N_SIGNAL   = 20    # features 0..19 carry true signal

# True coefficients: non-zero only for first N_SIGNAL features
true_beta    = np.zeros(N_FEATURES)
true_beta[:N_SIGNAL] = rng.normal(0, 0.05, N_SIGNAL)

X_raw  = rng.standard_normal((N_SAMPLES, N_FEATURES))
# Introduce mild multicollinearity in signal features
common = rng.standard_normal(N_SAMPLES)
X_raw[:, :N_SIGNAL] += 0.4 * common[:, None]

y = X_raw @ true_beta + rng.normal(0, 0.08, N_SAMPLES)

SPLIT = int(0.7 * N_SAMPLES)
X_tr, X_te = X_raw[:SPLIT], X_raw[SPLIT:]
y_tr, y_te = y[:SPLIT],     y[SPLIT:]

scaler = StandardScaler().fit(X_tr)
Xtr_s, Xte_s = scaler.transform(X_tr), scaler.transform(X_te)

models = {
    'OLS':         LinearRegression(),
    'Ridge':       Ridge(alpha=1.0),
    'Lasso':       Lasso(alpha=0.003, max_iter=5000),
    'ElasticNet':  ElasticNet(alpha=0.003, l1_ratio=0.5, max_iter=5000),
}

results = {}
for name, mdl in models.items():
    mdl.fit(Xtr_s, y_tr)
    tr_r2 = r2_score(y_tr, mdl.predict(Xtr_s))
    te_r2 = r2_score(y_te, mdl.predict(Xte_s))
    coef  = mdl.coef_
    n_nonzero = np.sum(np.abs(coef) > 1e-4)
    results[name] = dict(train_r2=tr_r2, test_r2=te_r2,
                         coef=coef, n_nonzero=n_nonzero)
    print(f'{name:12s}  train R²={tr_r2:+.4f}  test R²={te_r2:+.4f}  non-zero coefs={n_nonzero}')

OLS           train R²=+0.9073  test R²=+0.8859  non-zero coefs=50
Ridge         train R²=+0.9073  test R²=+0.8860  non-zero coefs=50
Lasso         train R²=+0.9014  test R²=+0.8877  non-zero coefs=32
ElasticNet    train R²=+0.9055  test R²=+0.8877  non-zero coefs=39


In [5]:
# ── Plot: coefficient paths + R² comparison ──────────────────────────────────
fig = ps.make_subplots(
    rows=1, cols=2, horizontal_spacing=0.10,
    subplot_titles=[
        'Estimated Coefficients by Model (true non-zero = features 0–19)',
        'Train vs Test R² — Overfitting Gap'
    ]
)

COLORS = {'OLS':'#ff6b6b', 'Ridge':'cornflowerblue',
          'Lasso':'#00d4aa', 'ElasticNet':'#f0c040'}

feat_idx = np.arange(N_FEATURES)
for name, res in results.items():
    fig.add_trace(
        go.Scatter(x=feat_idx, y=res['coef'], mode='markers',
                   name=name, marker=dict(color=COLORS[name], size=5, opacity=0.8)),
        row=1, col=1
    )

# Mark true signal boundary
fig.add_vline(x=N_SIGNAL - 0.5, line_color='white', line_dash='dash',
              annotation_text='noise→', annotation_position='top right', row=1, col=1)

# R² bars
names = list(results.keys())
tr_r2 = [results[n]['train_r2'] for n in names]
te_r2 = [results[n]['test_r2']  for n in names]
fig.add_trace(go.Bar(x=names, y=tr_r2, name='Train R²',
                     marker_color='#00d4aa', opacity=0.7), row=1, col=2)
fig.add_trace(go.Bar(x=names, y=te_r2, name='Test R²',
                     marker_color='#ff6b6b', opacity=0.85), row=1, col=2)

fig.update_layout(
    template='plotly_dark', height=450, barmode='group',
    title='Regularisation: Feature Selection & Generalisation'
)
fig.update_xaxes(title_text='Feature index', row=1, col=1)
fig.update_yaxes(title_text='Coefficient', row=1, col=1)
fig.update_yaxes(title_text='R²', row=1, col=2)
fig.show()

In [6]:
# ── Lasso regularisation path (lambda sweep) ──────────────────────────────────
alphas     = np.logspace(-4, 0, 60)
coef_paths = np.zeros((len(alphas), N_FEATURES))

for i, a in enumerate(alphas):
    mdl = Lasso(alpha=a, max_iter=10000).fit(Xtr_s, y_tr)
    coef_paths[i] = mdl.coef_

fig = go.Figure()
for j in range(N_FEATURES):
    color = '#00d4aa' if j < N_SIGNAL else 'rgba(200,80,80,0.4)'
    name  = f'feat {j} (signal)' if j < N_SIGNAL else (f'feat {j} (noise)' if j == N_SIGNAL else None)
    show  = j < N_SIGNAL or j == N_SIGNAL  # only first noise trace in legend
    fig.add_trace(go.Scatter(
        x=np.log10(alphas), y=coef_paths[:, j],
        mode='lines', line=dict(color=color, width=1.2),
        name=name, showlegend=show
    ))

fig.update_layout(
    template='plotly_dark', height=420,
    title='Lasso Regularisation Path — Coefficients vs log₁₀(λ)',
    xaxis_title='log₁₀(λ)  →  more regularisation',
    yaxis_title='Coefficient value',
    legend=dict(x=0.01, y=0.99)
)
fig.show()
print('Green lines = true signal features | Red lines = noise features')
print('As λ increases, noise features are zeroed out first.')

Green lines = true signal features | Red lines = noise features
As λ increases, noise features are zeroed out first.


**Key observations:**
- OLS assigns large coefficients to noise features, inflating train R².
- Lasso zeros out noise features; the regularisation path shows signal features surviving longer.
- Elastic Net combines sparsity (Lasso) with stability under multicollinearity (Ridge).

---
## Part 3 | Parameter Stability

### 3a — Sensitivity Surface (The 'Flat Top' Principle)

We grid-search `(fast, slow)` MA windows on the full in-sample and plot the Sharpe heatmap.  
**Robust parameters live on stable plateaus, not isolated spikes.**

$$\text{Plateau score} = \frac{\mu_\text{region}}{\sigma_\text{region}}$$

In [7]:
# ── Full parameter surface ─────────────────────────────────────────────────────
IS_RET = ret_series.iloc[:IS_BARS * 3]   # 3-year IS window

FAST_RANGE = range(2, 25)
SLOW_RANGE = range(10, 90, 3)

surface = np.full((len(FAST_RANGE), len(SLOW_RANGE)), np.nan)
for fi, f in enumerate(FAST_RANGE):
    for si, s in enumerate(SLOW_RANGE):
        if f >= s:
            continue
        surface[fi, si] = ma_sharpe(IS_RET, f, s)

fig = go.Figure(go.Heatmap(
    z=surface,
    x=list(SLOW_RANGE),
    y=list(FAST_RANGE),
    colorscale='RdYlGn',
    colorbar=dict(title='Sharpe Ratio'),
    zmin=-1.5, zmax=1.5
))

fig.update_layout(
    template='plotly_dark', height=450,
    title='Parameter Sensitivity Surface: MA Crossover (fast vs slow window)',
    xaxis_title='Slow window (bars)',
    yaxis_title='Fast window (bars)'
)
fig.show()

# Find the best parameter set and its neighbours
flat    = np.nanargmax(surface)
fi_best = flat // len(SLOW_RANGE)
si_best = flat % len(SLOW_RANGE)
f_best  = list(FAST_RANGE)[fi_best]
s_best  = list(SLOW_RANGE)[si_best]
print(f'Best IS params: fast={f_best}, slow={s_best}  Sharpe={surface[fi_best, si_best]:.3f}')

# Local neighbourhood Sharpe (±2 steps)
pad = 2
fi0, fi1 = max(0, fi_best-pad), min(surface.shape[0], fi_best+pad+1)
si0, si1 = max(0, si_best-pad), min(surface.shape[1], si_best+pad+1)
nbhd     = surface[fi0:fi1, si0:si1]
print(f'Neighbourhood mean SR: {np.nanmean(nbhd):.3f}  std: {np.nanstd(nbhd):.3f}')
print('(High std → fragile spike; low std → robust plateau)')

Best IS params: fast=14, slow=88  Sharpe=1.849
Neighbourhood mean SR: 1.602  std: 0.121
(High std → fragile spike; low std → robust plateau)


### 3b — Monte Carlo Permutation Test

Shuffle the return series to destroy any temporal signal, re-run the *same* optimisation, and record the best Sharpe.  
Repeat 1 000 times to build the **null distribution**.

$$p\text{-value} = \frac{\#\{\text{shuffled SR} \geq \text{observed SR}\}}{N_{\text{trials}}}$$

In [8]:
# ── Monte Carlo permutation test ──────────────────────────────────────────────
N_PERM   = 1000
OBS_SR   = surface[fi_best, si_best]     # observed best IS Sharpe

# Quick grid (coarse) for speed
F_QUICK = [3, 5, 10, 15]
S_QUICK = [20, 40, 60, 80]

null_best_srs = []
for _ in range(N_PERM):
    shuffled = IS_RET.sample(frac=1).values   # destroy time structure
    shuffled_s = pd.Series(shuffled, index=IS_RET.index)
    best = max(
        ma_sharpe(shuffled_s, f, s)
        for f in F_QUICK for s in S_QUICK if f < s
    )
    null_best_srs.append(best)

null_arr = np.array(null_best_srs)
p_value  = (null_arr >= OBS_SR).mean()

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=null_arr, nbinsx=50,
    name='Null distribution (shuffled)',
    marker_color='#4a6fa5', opacity=0.75
))
fig.add_vline(
    x=OBS_SR, line_color='#00d4aa', line_width=2, line_dash='dash',
    annotation_text=f'Observed SR={OBS_SR:.2f}',
    annotation_position='top left'
)

fig.update_layout(
    template='plotly_dark', height=400,
    title=f'Monte Carlo Permutation Test  |  p-value = {p_value:.3f}  ({N_PERM} trials)',
    xaxis_title='Best Sharpe under null (no signal)',
    yaxis_title='Count',
    legend=dict(x=0.01, y=0.99)
)
fig.show()

print(f'Observed best Sharpe : {OBS_SR:.3f}')
print(f'Null 95th percentile : {np.percentile(null_arr, 95):.3f}')
print(f'p-value              : {p_value:.3f}')
print(f'Skill signal         : {"YES" if p_value < 0.05 else "INCONCLUSIVE"}')

Observed best Sharpe : 1.849
Null 95th percentile : 1.908
p-value              : 0.067
Skill signal         : INCONCLUSIVE


### 3c — Bootstrap Stability: Parameter Variance

Re-sample the IS data **with replacement** to create synthetic histories.  
Each bootstrap selects a *different* best `(fast, slow)` pair — the spread quantifies parameter uncertainty.

In [9]:
# ── Bootstrap stability ───────────────────────────────────────────────────────
N_BOOT     = 500
boot_fast, boot_slow, boot_sr = [], [], []

for _ in range(N_BOOT):
    idx    = rng.integers(0, len(IS_RET), size=len(IS_RET))
    b_ret  = pd.Series(IS_RET.values[idx], index=IS_RET.index)
    best_b, bf, bs = -99, F_QUICK[0], S_QUICK[-1]
    for f in F_QUICK:
        for s in S_QUICK:
            if f >= s:
                continue
            sr = ma_sharpe(b_ret, f, s)
            if sr > best_b:
                best_b, bf, bs = sr, f, s
    boot_fast.append(bf)
    boot_slow.append(bs)
    boot_sr.append(best_b)

boot_fast = np.array(boot_fast)
boot_slow = np.array(boot_slow)

fig = ps.make_subplots(
    rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=[
        'Bootstrap: Distribution of Optimal Fast Window',
        'Bootstrap: Distribution of Optimal Slow Window'
    ]
)
fig.add_trace(go.Histogram(x=boot_fast, nbinsx=10, name='Fast window',
                            marker_color='#00d4aa', opacity=0.8), row=1, col=1)
fig.add_trace(go.Histogram(x=boot_slow, nbinsx=10, name='Slow window',
                            marker_color='cornflowerblue', opacity=0.8), row=1, col=2)

fig.update_layout(
    template='plotly_dark', height=380,
    title='Bootstrap Parameter Stability  (wide spread = fragile optimum)',
    showlegend=False
)
fig.update_xaxes(title_text='Bars', row=1, col=1)
fig.update_xaxes(title_text='Bars', row=1, col=2)
fig.show()

from collections import Counter
print('Fast window frequencies:', dict(Counter(boot_fast)))
print('Slow window frequencies:', dict(Counter(boot_slow)))

Fast window frequencies: {np.int64(5): 103, np.int64(3): 116, np.int64(15): 165, np.int64(10): 116}
Slow window frequencies: {np.int64(60): 98, np.int64(40): 80, np.int64(80): 225, np.int64(20): 97}


---
## Part 4 | Statistical Testing

### 4a — Probabilistic Sharpe Ratio (PSR)

The PSR asks: *given finite sample size and non-normal returns, what is the probability that the true Sharpe exceeds a benchmark SR\*?*

$$\text{PSR}(\text{SR}^*) = \Phi\!\left[\frac{(\hat{SR} - \text{SR}^*)\sqrt{T-1}}{\sqrt{1 - \hat{\gamma}_3 \hat{SR} + \frac{\hat{\gamma}_4 - 1}{4}\hat{SR}^2}}\right]$$

where $\hat{\gamma}_3$ is skewness and $\hat{\gamma}_4$ is excess kurtosis of the returns.

In [10]:
# ── PSR and DSR implementations ───────────────────────────────────────────────
def sharpe_annualised(r):
    r = np.asarray(r)
    return r.mean() / r.std() * ANNUAL**0.5

def psr(returns_arr, sr_star=0.0):
    """
    Probabilistic Sharpe Ratio.
    Returns the probability that the true SR > sr_star.
    """
    r   = np.asarray(returns_arr)
    n   = len(r)
    sr  = r.mean() / r.std()              # daily Sharpe
    sk  = stats.skew(r)
    ku  = stats.kurtosis(r)               # excess kurtosis
    sr_bench = sr_star / ANNUAL**0.5      # convert to daily
    denom = np.sqrt(1 - sk * sr + (ku / 4) * sr**2)
    if denom <= 0:
        return np.nan
    z = (sr - sr_bench) * np.sqrt(n - 1) / denom
    return float(ndtr(z))

def dsr(returns_arr, n_trials, sr_star=0.0):
    """
    Deflated Sharpe Ratio.
    Adjusts SR* upward to account for multiple-testing bias,
    then returns PSR against that higher benchmark.
    Approximation from Bailey & Lopez de Prado (2014).
    """
    r     = np.asarray(returns_arr)
    n     = len(r)
    # Expected maximum SR under H0 (Gaussian approximation)
    euler = 0.5772156649
    if n_trials > 1:
        sr0 = (1 - euler) * stats.norm.ppf(1 - 1/n_trials) + \
              euler       * stats.norm.ppf(1 - 1/(n_trials * np.e))
    else:
        sr0 = 0
    # sr0 is in daily units; psr expects annualised sr_star
    sr0_ann = sr0 * ANNUAL**0.5
    return psr(r, sr_star=sr0_ann), sr0_ann

# ── Apply to OOS PnL series ───────────────────────────────────────────────────
oos = oos_pnl_series.dropna().values
n_strategies_tested = len(FAST_GRID) * len(SLOW_GRID)   # grid size

obs_sr   = sharpe_annualised(oos)
psr_val  = psr(oos, sr_star=0.0)
dsr_val, sr0_ann = dsr(oos, n_trials=n_strategies_tested)

print(f'OOS observations    : {len(oos)}')
print(f'Observed ann. SR    : {obs_sr:+.3f}')
print(f'Skewness            : {stats.skew(oos):+.3f}')
print(f'Excess kurtosis     : {stats.kurtosis(oos):+.3f}')
print()
print(f'PSR(SR*=0)          : {psr_val:.3f}  (prob true SR > 0)')
print(f'Adjusted SR* (DSR)  : {sr0_ann:.3f}  (expected max under {n_strategies_tested} trials)')
print(f'DSR                 : {dsr_val:.3f}  (prob true SR > adjusted benchmark)')

OOS observations    : 1008
Observed ann. SR    : +1.453
Skewness            : +0.039
Excess kurtosis     : +0.009

PSR(SR*=0)          : 0.998  (prob true SR > 0)
Adjusted SR* (DSR)  : 31.704  (expected max under 25 trials)
DSR                 : 0.000  (prob true SR > adjusted benchmark)


In [11]:
# ── PSR curve across SR* thresholds ──────────────────────────────────────────
sr_stars = np.linspace(-0.5, 2.0, 200)
psr_vals = [psr(oos, s) for s in sr_stars]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sr_stars, y=psr_vals, mode='lines',
    line=dict(color='#00d4aa', width=2.5), name='PSR'
))
fig.add_vline(x=obs_sr, line_color='#f0c040', line_dash='dot',
              annotation_text=f'Observed SR={obs_sr:.2f}', annotation_position='top left')
fig.add_vline(x=sr0_ann, line_color='#ff6b6b', line_dash='dash',
              annotation_text=f'DSR benchmark={sr0_ann:.2f}', annotation_position='top right')
fig.add_hline(y=0.95, line_color='grey', line_dash='dot',
              annotation_text='95% confidence', annotation_position='right')

fig.update_layout(
    template='plotly_dark', height=420,
    title='Probabilistic Sharpe Ratio — Confidence vs Benchmark SR*',
    xaxis_title='Benchmark SR* (annualised)',
    yaxis_title='P(true SR > SR*)',
    yaxis_range=[0, 1]
)
fig.show()

### 4b — White's Reality Check & Haircut Sharpe

**White's Reality Check (WRC)**: when selecting the *best* of $M$ strategies, the null hypothesis is that *no* strategy has positive expected returns.

$$H_0: \max_{k=1..M}\, \mathbb{E}[f_k] \leq 0$$

We approximate via bootstrap: resample the time series of each candidate's returns, recompute the maximum performance, and compare the observed maximum to that null distribution.

**Haircut Sharpe**: a simple practical discount:
$$\text{SR}_{\text{haircut}} = \text{SR}_{\text{observed}} \times (1 - \text{discount})$$

In [12]:
# ── Collect candidate strategy returns (coarse grid on IS) ────────────────────
candidate_pnls = {}
price_is = (1 + IS_RET).cumprod()

for f in F_QUICK:
    for s in S_QUICK:
        if f >= s:
            continue
        ma_f  = price_is.rolling(f).mean()
        ma_s  = price_is.rolling(s).mean()
        sig   = np.sign(ma_f - ma_s).shift(1)
        pnl   = (sig * IS_RET).dropna()
        candidate_pnls[f'f{f}_s{s}'] = pnl.values

cand_names = list(candidate_pnls.keys())
# Stack only arrays of equal length (find min length)
min_len = min(len(candidate_pnls[k]) for k in cand_names)
cand_arr = np.vstack([candidate_pnls[k][:min_len] for k in cand_names])  # (M, T)
M          = cand_arr.shape[0]

# Observed maximum mean return
obs_means  = cand_arr.mean(axis=1)
obs_max    = obs_means.max()
best_idx   = obs_means.argmax()

print(f'Candidates: {M}')
print(f'Best strategy  : {cand_names[best_idx]}  mean daily ret={obs_max:.6f}')

# ── WRC bootstrap ────────────────────────────────────────────────────────────
N_WRC    = 2000
T_cand   = cand_arr.shape[1]
wrc_null = np.zeros(N_WRC)

# Demean each candidate (WRC tests H0: all means <= 0)
demeaned = cand_arr - cand_arr.mean(axis=1, keepdims=True)

for b in range(N_WRC):
    idx    = rng.integers(0, T_cand, size=T_cand)          # resample time
    boot_m = demeaned[:, idx].mean(axis=1)                 # max of demeaned
    wrc_null[b] = boot_m.max()

wrc_pval = (wrc_null >= obs_max).mean()

fig = go.Figure()
fig.add_trace(go.Histogram(x=wrc_null, nbinsx=60, name='WRC null',
                            marker_color='#4a6fa5', opacity=0.75))
fig.add_vline(x=obs_max, line_color='#00d4aa', line_width=2, line_dash='dash',
              annotation_text=f'Best observed={obs_max:.5f}',
              annotation_position='top left')

fig.update_layout(
    template='plotly_dark', height=380,
    title=f"White's Reality Check  |  p-value = {wrc_pval:.3f}  (M={M} strategies, {N_WRC} bootstraps)",
    xaxis_title='Max mean return under null',
    yaxis_title='Count'
)
fig.show()
print(f'WRC p-value: {wrc_pval:.3f}')

Candidates: 16
Best strategy  : f15_s80  mean daily ret=0.001077


WRC p-value: 0.033


In [13]:
# ── Haircut Sharpe ────────────────────────────────────────────────────────────
best_pnl    = cand_arr[best_idx]
raw_sr      = sharpe_annualised(best_pnl)

# Haircut components (illustrative)
trial_discount  = 1 - 1 / np.log(max(M, 2))     # more trials → bigger cut
complexity_disc = 0.90                            # 10% for 2 params
length_disc     = min(1.0, T_cand / (252 * 3))   # penalise < 3 years
total_haircut   = 1 - trial_discount * complexity_disc * length_disc
haircut_sr      = raw_sr * (1 - total_haircut)

labels = ['Raw IS SR', 'After trial-count', 'After complexity', 'After data-length (Haircut SR)']
values = [
    raw_sr,
    raw_sr * trial_discount,
    raw_sr * trial_discount * complexity_disc,
    haircut_sr
]

fig = go.Figure(go.Waterfall(
    orientation='v',
    measure=['absolute', 'relative', 'relative', 'total'],
    x=labels,
    y=[values[0], values[1]-values[0], values[2]-values[1], haircut_sr - values[0]],
    text=[f'{v:.3f}' for v in values],
    textposition='outside',
    connector=dict(line=dict(color='grey', dash='dot')),
    decreasing=dict(marker_color='#ff6b6b'),
    increasing=dict(marker_color='#00d4aa'),
    totals=dict(marker_color='#f0c040')
))
fig.update_layout(
    template='plotly_dark', height=420,
    title=f'Haircut Sharpe Ratio  |  Raw={raw_sr:.2f} → Haircut={haircut_sr:.2f}',
    yaxis_title='Sharpe Ratio'
)
fig.show()

---
## Part 5 | Monitoring & The Kill Switch

Once deployed, a strategy must be continuously monitored for **structural drift**.  
We define three kill-switch triggers:

| Trigger | Threshold |
|---------|----------|
| Max Drawdown Breach | Live DD > 1.2× backtest max DD |
| Sharpe Ratio Decay | Rolling 63-day SR < 50% of target |
| Regime Shift | 21-day realised vol > 2× backtest vol |

We simulate a live period where the regime shifts halfway through, and show how the monitoring dashboard would detect it.

$$Z_t = \frac{R_{\text{live},t} - \mu_{\text{backtest}}}{\sigma_{\text{backtest}}}$$

In [14]:
# ── Simulate live period with mid-point regime shift ─────────────────────────
LIVE_BARS   = 252
SHIFT_BAR   = 126      # regime shift at halfway

# Phase 1: strategy works (same DGP)
live_ph1 = TRUE_ALPHA * true_signal[:SHIFT_BAR] + DAILY_VOL * rng.standard_normal(SHIFT_BAR)
# Phase 2: alpha decays, vol spikes
live_ph2 = 0.0 * true_signal[:LIVE_BARS - SHIFT_BAR] + \
           DAILY_VOL * 2.0 * rng.standard_normal(LIVE_BARS - SHIFT_BAR)

live_rets  = np.concatenate([live_ph1, live_ph2])
live_dates = pd.date_range('2024-01-02', periods=LIVE_BARS, freq='B')
live_ser   = pd.Series(live_rets, index=live_dates)

# Backtest reference statistics
bt_mu    = IS_RET.mean()
bt_sigma = IS_RET.std()
bt_sr    = bt_mu / bt_sigma * ANNUAL**0.5
bt_vol   = bt_sigma

# ── Rolling metrics ───────────────────────────────────────────────────────────
ROLL_VOL = 21
ROLL_SR  = 63

live_eq   = (1 + live_ser).cumprod()
roll_dd   = (live_eq / live_eq.cummax() - 1)          # rolling drawdown
roll_vol  = live_ser.rolling(ROLL_VOL).std() * ANNUAL**0.5
roll_sr   = live_ser.rolling(ROLL_SR).apply(
    lambda x: x.mean() / x.std() * ANNUAL**0.5 if x.std() > 0 else 0
)
perf_z    = (live_ser - bt_mu) / bt_sigma             # daily Z-score
cumul_z   = perf_z.cumsum()

# Kill-switch thresholds
bt_max_dd  = (IS_RET.pipe(lambda r: (1 + r).cumprod()).pipe(lambda eq: eq / eq.cummax() - 1)).min()
dd_thresh  = bt_max_dd * 1.2
sr_thresh  = bt_sr * 0.5
vol_thresh = bt_vol * 2.0 * ANNUAL**0.5

# Detect triggers
dd_breach  = roll_dd[roll_dd < dd_thresh].index
sr_breach  = roll_sr[roll_sr < sr_thresh].index
vol_breach = roll_vol[roll_vol > vol_thresh].index

print(f'Backtest SR    : {bt_sr:.3f}   |  Kill SR threshold  : {sr_thresh:.3f}')
print(f'Backtest MaxDD : {bt_max_dd:.3f}   |  Kill DD threshold  : {dd_thresh:.3f}')
print(f'Backtest ann.vol: {bt_vol*ANNUAL**0.5:.3f}  |  Kill vol threshold : {vol_thresh:.3f}')
print()
print(f'Max DD breach   : first at {dd_breach[0].date() if len(dd_breach) else "none"}')
print(f'SR decay breach : first at {sr_breach[0].date() if len(sr_breach) else "none"}')
print(f'Vol spike breach: first at {vol_breach[0].date() if len(vol_breach) else "none"}')

Backtest SR    : -1.643   |  Kill SR threshold  : -0.821
Backtest MaxDD : -0.630   |  Kill DD threshold  : -0.755
Backtest ann.vol: 0.162  |  Kill vol threshold : 0.325

Max DD breach   : first at none
SR decay breach : first at 2024-08-12
Vol spike breach: first at 2024-08-02


In [19]:
# ── Monitoring dashboard ──────────────────────────────────────────────────────
shift_date = live_dates[SHIFT_BAR]

fig = ps.make_subplots(
    rows=4, cols=1, vertical_spacing=0.06, shared_xaxes=True,
    subplot_titles=[
        'Live Equity Curve',
        'Drawdown  (kill if < 1.2× backtest max DD)',
        '63-day Rolling Sharpe  (kill if < 50% target)',
        '21-day Realised Vol  (kill if > 2× backtest vol)'
    ]
)

def add_regime_shade(fig, rows):
    for r in rows:
        fig.add_vrect(
            x0=str(shift_date), x1=str(live_dates[-1]),  # str() avoids Timestamp arithmetic error
            fillcolor='rgba(255,107,107,0.10)', line_width=0,
            row=r, col=1
        )

# Equity
fig.add_trace(go.Scatter(x=live_eq.index, y=live_eq.values, mode='lines',
                          name='Equity', line=dict(color='#00d4aa', width=2)), row=1, col=1)

# Drawdown
fig.add_trace(go.Scatter(x=roll_dd.index, y=roll_dd.values, mode='lines',
                          name='Drawdown', line=dict(color='#ff6b6b', width=1.5),
                          fill='tozeroy', fillcolor='rgba(255,107,107,0.15)'), row=2, col=1)
fig.add_hline(y=dd_thresh, line_color='#f0c040', line_dash='dash', row=2, col=1)

# Rolling SR
fig.add_trace(go.Scatter(x=roll_sr.index, y=roll_sr.values, mode='lines',
                          name='Rolling SR', line=dict(color='cornflowerblue', width=1.5)), row=3, col=1)
fig.add_hline(y=sr_thresh, line_color='#f0c040', line_dash='dash', row=3, col=1)
fig.add_hline(y=0, line_color='grey', line_width=0.8, row=3, col=1)

# Rolling vol
fig.add_trace(go.Scatter(x=roll_vol.index, y=roll_vol.values, mode='lines',
                          name='Realised Vol', line=dict(color='#f0c040', width=1.5)), row=4, col=1)
fig.add_hline(y=vol_thresh, line_color='#ff6b6b', line_dash='dash', row=4, col=1)

# Regime shift marker — use add_shape instead of add_vline to avoid Timestamp arithmetic error
for r in [1, 2, 3, 4]:
    fig.add_shape(
        type='line', xref='x', yref='paper',
        x0=shift_date, x1=shift_date, y0=0, y1=1,
        line=dict(color='white', dash='dot', width=1),
        row=r, col=1
    )
fig.add_annotation(
    x=shift_date, xref='x', yref='paper', y=0.97,
    text='Regime shift', showarrow=False,
    font=dict(color='white', size=10), row=1, col=1
)

add_regime_shade(fig, [1, 2, 3, 4])

fig.update_layout(
    template='plotly_dark', height=800, showlegend=False,
    title='Live Monitoring Dashboard — Kill Switch Triggers (dashed yellow lines)'
)
fig.show()

In [24]:
# ── CUSUM structural break detection ─────────────────────────────────────────
# CUSUM of standardised daily returns: persistent deviation signals a break
cusum      = ((live_ser - bt_mu) / bt_sigma).cumsum()
cusum_pos  = cusum.clip(lower=0)
cusum_neg  = cusum.clip(upper=0)

# Control limits at ±3σ of the expected walk
cusum_limit = 3 * np.sqrt(np.arange(1, LIVE_BARS + 1))

fig = go.Figure()
fig.add_trace(go.Scatter(x=live_dates, y=cusum.values, mode='lines',
                          name='CUSUM', line=dict(color='#00d4aa', width=2)))
fig.add_trace(go.Scatter(x=live_dates, y=cusum_limit,  mode='lines',
                          name='+3σ limit', line=dict(color='#ff6b6b', dash='dash', width=1.2)))
fig.add_trace(go.Scatter(x=live_dates, y=-cusum_limit, mode='lines',
                          name='−3σ limit', line=dict(color='#ff6b6b', dash='dash', width=1.2),
                          showlegend=False))
fig.add_shape(
    type='line', xref='x', yref='paper',
    x0=shift_date, x1=shift_date, y0=0, y1=1,
    line=dict(color='white', dash='dot', width=1)
)
fig.add_annotation(
    x=shift_date, xref='x', yref='paper', y=0.97,
    text='Regime shift', showarrow=False,
    font=dict(color='white', size=10), xanchor='left'
)
fig.add_vrect(x0=str(shift_date), x1=str(live_dates[-1]),
              fillcolor='rgba(255,107,107,0.08)', line_width=0)

fig.update_layout(
    template='plotly_dark', height=400,
    title='CUSUM Structural Break Detector — Cumulative Standardised Returns',
    xaxis_title='Date',
    yaxis_title='Cumulative Z-score'
)
fig.show()
print('CUSUM drifting outside limits signals the strategy has broken.')

CUSUM drifting outside limits signals the strategy has broken.


---
## Summary — The Robustness Audit Checklist

| # | Check | Tool used in this notebook |
|---|-------|-----------------------------|
| 1 | Data Integrity (no look-ahead) | 1-bar lagged signals throughout |
| 2 | Leakage Check | OOS data strictly isolated per fold |
| 3 | Walk-Forward Stability | IS vs OOS Sharpe per fold |
| 4 | Parameter Sensitivity | 2D Sharpe surface + flat-top analysis |
| 5 | Transaction Costs | (apply slippage haircut to `pnl_oos`) |
| 6 | Statistical Significance | PSR, DSR, White's Reality Check |
| 7 | Regime Robustness | Monte Carlo permutation |
| 8 | Complexity Penalty | Haircut Sharpe, Lasso L1 regularisation |
| 9 | Execution Capacity | (scale `live_pnl` by capacity assumption) |
| 10 | Economic Logic | MA-crossover captures momentum — a known persistent anomaly |

> *"Define a kill switch before deployment, not during a drawdown."*